
<div style="text-align: center; line-height: 0; padding-top: 9px;">
  <img
    src="https://databricks.com/wp-content/uploads/2018/03/db-academy-rgb-1200px.png"
    alt="Databricks Learning"
  >
</div>

# Demo - Agent Setup
## Overview
In this demo, we'll set up a tool-calling agent registered to Unity Catalog and verify it works by generating MLflow traces. This agent will be used throughout the remaining demos and labs for evaluation exercises.

## Learning Objectives
By the end of this demo, you will be able to:
1. Load a Unity Catalog-registered agent by name and alias
2. Generate traces by interacting with the agent
3. Locate those traces in Unity Catalog

<div style="border-left: 4px solid #f44336; background: #ffebee; padding: 16px 20px; border-radius: 4px; margin: 16px 0;">
<div style="display: flex; align-items: flex-start; gap: 12px;">
<div>
<strong style="color: #c62828; font-size: 1.1em;">Select Compute</strong>
<p style="margin: 8px 0 0 0; color: #333;">Before starting this notebook, select the required compute environment listed below.</p>
<ul style="margin: 12px 0 0 16px; color: #333;">
<li><strong>Serverless Compute, Version 5</strong>: How to select an environment version (<a href="https://docs.databricks.com/aws/en/compute/serverless/dependencies#-select-an-environment-version" style="color: #1976d2; text-decoration: underline;">AWS</a> | <a href="https://learn.microsoft.com/en-us/azure/databricks/compute/serverless/dependencies#-select-an-environment-version" style="color: #1976d2; text-decoration: underline;">Azure</a> | <a href="https://docs.databricks.com/gcp/en/compute/serverless/dependencies#-select-an-environment-version" style="color: #1976d2; text-decoration: underline;">GCP</a>)</li>
</ul>
<p style="margin: 8px 0 0 0; color: #333;"><strong>NOTE:</strong> This notebook was <strong>developed and tested using Serverless V5</strong>. Other compute options may work but are not guaranteed to behave the same or support all features demonstrated.</p>
</div>
</div>
</div>

## A. Classroom Setup
Run the following cell to configure your environment. This provisions the catalog, schema, data, tools, agents, and evaluation datasets.

In [0]:
%run ../Includes/Classroom-Setup-1

Using Python 3.12.3 environment at: /local_disk0/.ephemeral_nfs/envs/pythonEnv-2aecede8-0673-4dd2-b385-a1255b6b95d9
Resolved 1 package in 198ms
Prepared 1 package in 119ms
Installed 1 package in 1ms
 + backoff==2.2.1
Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.
Using Python 3.12.3 environment at: /local_disk0/.ephemeral_nfs/envs/pythonEnv-2aecede8-0673-4dd2-b385-a1255b6b95d9
Resolved 129 packages in 2.06s
 Downloaded pydantic-core
 Downloaded sqlalchemy
 Downloaded openai-agents
 Downloaded mlflow-tracing
 Downloaded databricks-connect
 Downloaded mlflow-skinny
 Downloaded mlflow
 Downloaded openai
Prepared 59 packages in 1.17s
Uninstalled 20 packages in 40ms
Installed 59 packages in 298ms
 + aiohttp-retry==2.9.1
 + alembic==1.20.0
 - anyio==4.7.0
 + anyio==4.15.1
 - blinker==1.7.0
 + blinker==1.9.0
 + databricks-ai-bridge==0.22.0
 + databricks-ai-search==0.79
 - databricks-connect==18.1.9
 + databricks-conne

Pre-installed Databricks Runtime package(s) replaced: starlette: 0.50.0 -> 1.9.0, idna: 3.7 -> 3.20, opentelemetry-sdk: 1.39.1 -> 1.45.1, opentelemetry-semantic-conventions: 0.60b1 -> 0.66b1, protobuf: 5.29.4 -> 5.29.6, databricks-connect: 18.1.9 -> 17.0.10, mlflow-skinny: 3.8.1 -> 3.17.1, openai: 2.14.0 -> 3.35.1, typing_extensions: 4.12.2 -> 4.16.0, fastapi: 0.128.0 -> 0.146.0, pydantic: 2.10.6 -> 2.14.1, blinker: 1.7.0 -> 1.9.0, pydantic_core: 2.27.2 -> 2.50.1, jiter: 0.12.0 -> 0.17.0, opentelemetry-api: 1.39.1 -> 1.45.1, PyJWT: 2.10.1 -> 2.15.1, wcwidth: 0.2.5 -> 0.9.2, anyio: 4.7.0 -> 4.15.1, requests: 2.32.3 -> 2.34.2, urllib3: 2.3.0 -> 2.8.0. uv pip install may report the previous version as uninstalled, but the runtime's preinstalled packages remain on disk.


Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.
Using Python 3.12.3 environment at: /local_disk0/.ephemeral_nfs/envs/pythonEnv-2aecede8-0673-4dd2-b385-a1255b6b95d9
Resolved 1 package in 214ms
 Downloaded uv
Prepared 1 package in 505ms
Installed 1 package in 82ms
 + uv==0.11.16
Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.
Using Python 3.12.3 environment at: /local_disk0/.ephemeral_nfs/envs/pythonEnv-2aecede8-0673-4dd2-b385-a1255b6b95d9
Resolved 62 packages in 1.07s
Prepared 1 package in 76ms
Uninstalled 1 package in 1ms
Installed 1 package in 14ms
 - databricks-agents==1.9.1
 + databricks-agents==1.10.2


Pre-installed Databricks Runtime package(s) replaced: databricks-agents: 1.9.1 -> 1.10.2. uv pip install may report the previous version as uninstalled, but the runtime's preinstalled packages remain on disk.


Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.
Using Python 3.12.3 environment at: /local_disk0/.ephemeral_nfs/envs/pythonEnv-2aecede8-0673-4dd2-b385-a1255b6b95d9
Resolved 75 packages in 747ms
 Downloaded mlflow-skinny
Prepared 2 packages in 262ms
Uninstalled 2 packages in 25ms
Installed 2 packages in 96ms
 - mlflow-skinny==3.17.1
 + mlflow-skinny==3.12.0
 - starlette==1.9.0
 + starlette==0.52.1


...

  Model registered: labuser17704131_1792356850.demo_agent_eval.nyc_taxi (version 1, alias 'champion')
Deploying agent: labuser17704131_1792356850.demo_agent_eval.airbnb
Endpoint name: labuser17704131_1792356850_agent


If you are using MLflow Tracing, you can migrate your traces to Unity Catalog for unlimited storage, fine-grained access controls, and queryability from notebooks, SQL, and dashboards. Learn more: https://docs.databricks.com/aws/en/mlflow3/genai/tracing/migrate-traces-to-uc


  Retrieved model version 1 with alias 'champion'


/local_disk0/.ephemeral_nfs/envs/pythonEnv-2aecede8-0673-4dd2-b385-a1255b6b95d9/lib/python3.12/site-packages/databricks/agents/deployments.py:641: UserWarning: This endpoint is being deployed without a feedback model, which has been deprecated.
For more information, see: https://docs.databricks.com/aws/en/generative-ai/agent-framework/feedback-model
  warnings.warn(



    Deployment of labuser17704131_1792356850.demo_agent_eval.airbnb version 1 initiated.  This can take up to 15 minutes and the Review App & Query Endpoint will not work until this deployment finishes.

    View status: https://dbc-4d4ce156-516a.cloud.databricks.com/ml/endpoints/labuser17704131_1792356850_agent/?o=7474659189051797
    Review App: https://dbc-4d4ce156-516a.cloud.databricks.com/ml/review-v2/ebb97dfe15874008bb81ab5c7575e602/chat?o=7474659189051797

You can refer back to the links above from the endpoint detail page at https://dbc-4d4ce156-516a.cloud.databricks.com/ml/endpoints/labuser17704131_1792356850_agent/?o=7474659189051797.

To set up monitoring for your deployed agent, see:
https://docs.databricks.com/aws/en/mlflow3/genai/eval-monitor/production-monitoring
  Deployed endpoint: labuser17704131_1792356850_agent

  Agent 'airbnb' deployed to endpoint: labuser17704131_1792356850_agent
Refreshing environment

Demo Environment Setup Complete


In [0]:
print(f"Username:          {username}")
print(f"Catalog Name:      {catalog_name}")
print(f"Schema Name:       {schema_name}")

Username:          labuser17704131_1792356850@vocareum.com
Catalog Name:      labuser17704131_1792356850
Schema Name:       demo_agent_eval


## B. Load the Agent

The setup script registered a tool-calling agent to Unity Catalog. We'll load it using `mlflow.pyfunc.load_model()` with the model's UC path and `@champion` alias.

<div style="border-left: 4px solid #1976d2; background: #e3f2fd; padding: 16px 20px; border-radius: 4px; margin: 16px 0;">
<div style="display: flex; align-items: flex-start; gap: 12px;">
<div>
<strong style="color: #0d47a1; font-size: 1.1em;">About This Agent</strong>
<p style="margin: 8px 0 0 0; color: #333;">This is a simple <strong>tool-calling agent</strong> backed by SQL UDFs registered to Unity Catalog. The agent itself is not the focus - it exists so we can practice evaluation techniques.</p>
</div>
</div>
</div>

In [0]:
import mlflow

mlflow.set_registry_uri("databricks-uc")
UC_MODEL_NAME = f"{catalog_name}.{schema_name}.airbnb"
agent = mlflow.pyfunc.load_model(f"models:/{UC_MODEL_NAME}@champion")

2026/10/18 21:15:28 WARNING mlflow.utils.requirements_utils: Detected one or more mismatches between the model's dependencies and the current Python environment:
 - mlflow (current: 3.17.1, required: mlflow==3.13.0)
 - pyyaml (current: 6.0.2, required: pyyaml==6.0.3)
To fix the mismatches, call `mlflow.pyfunc.get_model_dependencies(model_uri)` to fetch the model's environment and install dependencies using the resulting environment file.



## C. Query the Agent and Inspect Traces

Let's send some questions to the agent using its `predict()` method. Each call automatically generates an **MLflow trace**: a structured record of the agent's full execution flow, including:

- **Spans** for each step (LLM call, tool invocation, retrieval)
- **Input/output payloads** at every span
- **Latency** for each operation

Traces are the foundation for evaluation. They let us verify *what* the agent did, not just what it returned.

The helper below formats our question into the **ChatCompletionRequest** schema the agent expects: a list of message objects with `role` and `content` fields, matching the OpenAI-compatible message format used by MLflow's pyfunc serving interface.

In [0]:
def agent_payload(question: str):
    return [{"input": [{"role": "user", "content": question}]}]

This first question is a **metadata query**: the agent can answer it directly from its system prompt without invoking any tools. Notice that the resulting trace will show only an LLM span with no tool-call spans.

In [0]:
agent.predict(agent_payload("What tools do you have available?"))

/local_disk0/.ephemeral_nfs/envs/pythonEnv-2aecede8-0673-4dd2-b385-a1255b6b95d9/lib/python3.12/site-packages/pydantic/main.py:475: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `str` - serialized value may not be as expected [field_name='content', input_value=[{'type': 'reasoning', 's...ings for a room type.'}], input_type=list])
  return self.__pydantic_serializer__.to_python(


{'object': 'response',
 'output': [{'type': 'message',
   'id': 'chatcmpl_826fe685-96c2-4731-8915-de9dbf8fdb12_44baf079-21bc-443a-87c0-19d5fe7c7498',
   'content': [{'text': 'I have two specialized tools that work with Airbnb listing data:\n\n| Tool | Description |\n|------|-------------|\n| `labuser17704131_1792356850__demo_agent_eval__avg_neigh_price` | Calculates the average listing price for a specific neighborhood in San\u202fFrancisco. It cleans price strings and converts them to numeric values before computing the mean. |\n| `labuser17704131_1792356850__demo_agent_eval__cnt_by_room_type` | Counts the number of Airbnb listings in a given neighborhood that match a particular room type (e.g., “Private room” or “Shared room”). |\n\nUse these whenever you need neighborhood‑level statistics such as average price or the number of listings for a room type.',
     'type': 'output_text',
     'annotations': []}],
   'role': 'assistant'}]}

Trace(trace_id='tr-38d18694f5751d821a19442976f77c19')

The next question will trigger both `avg_neigh_price` and `cnt_by_room_type` tool calls. Inspect the trace output to see the tool invocations.

In [0]:
agent.predict(agent_payload("Can you tell me what the average price is in Mission? Also, what's the number of listings for that neighborhood that have private rooms?"))

/local_disk0/.ephemeral_nfs/envs/pythonEnv-2aecede8-0673-4dd2-b385-a1255b6b95d9/lib/python3.12/site-packages/pydantic/main.py:475: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `str` - serialized value may not be as expected [field_name='content', input_value=[{'type': 'reasoning', 's...rovide the results.'}]}], input_type=list])
  return self.__pydantic_serializer__.to_python(
/local_disk0/.ephemeral_nfs/envs/pythonEnv-2aecede8-0673-4dd2-b385-a1255b6b95d9/lib/python3.12/site-packages/pydantic/main.py:475: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `str` - serialized value may not be as expected [field_name='content', input_value=[{'type': 'reasoning', 's... for private rooms.'}]}], input_type=list])
  return self.__pydantic_serializer__.to_python(
/local_disk0/.ephemeral_nfs/envs/pythonEnv-2aecede8-0673-4dd2-b385-a1255b6b95d9/lib/python3.12/site-packages/pydantic/main.py:475: UserWarning: Pyda

{'object': 'response',
 'output': [{'type': 'function_call',
   'id': 'chatcmpl_8cfce7a6-d594-42f3-a06e-fd48f5575b31_f1ab0438-0035-44ee-bf7c-9eb3dac5e50f',
   'call_id': 'call_57320ab9-742b-4d44-9d40-a603eef97764',
   'name': 'labuser17704131_1792356850__demo_agent_eval__avg_neigh_price',
   'arguments': '{"neighborhood_name":"Mission"}'},
  {'type': 'function_call_output',
   'call_id': 'call_57320ab9-742b-4d44-9d40-a603eef97764',
   'output': '188.625'},
  {'type': 'function_call',
   'id': 'chatcmpl_2068e462-1357-4cb4-9a7b-4e80824f6ed9_0584f569-79e7-4754-a228-b7bafaf662c0',
   'call_id': 'call_51d4ce15-c492-40a0-ae7e-d1760481f0a3',
   'name': 'labuser17704131_1792356850__demo_agent_eval__cnt_by_room_type',
   'arguments': '{"neighborhood_name":"Mission","room_type_filter":"Private room"}'},
  {'type': 'function_call_output',
   'call_id': 'call_51d4ce15-c492-40a0-ae7e-d1760481f0a3',
   'output': '2'},
  {'type': 'message',
   'id': 'chatcmpl_51fba970-e9c1-4308-8dd2-9f892dca77a7_29c2

Trace(trace_id='tr-d49b2dd630babc104737dda21b10befc')


## D. View Traces in Unity Catalog

Navigate to **Catalog Explorer** and search for **airbnb**. You'll find the registered model with the alias **@champion**. Click the latest version (you will create multiple versions of the agent if you rerun the setup script but the **@champion** alias will always point to the latest registered version) and open the **Traces** tab to see the two requests we just sent.

**What to look for:**
- **Span hierarchy** - expand a trace to see nested spans (LLM → tool calls → tool responses)
- **Inputs & outputs** - click any span to inspect the exact payload sent and received
- **Latency breakdown** - identify which step (LLM reasoning vs. tool execution) took the most time
- **Contrast the two traces** - the first has no tool spans; the second shows two parallel tool invocations

## E. Conclusion
In this demo, you:

1. Loaded a Unity Catalog-registered agent using `mlflow.pyfunc.load_model()`
2. Queried the agent and observed MLflow traces capturing tool calls
3. Located traces in Unity Catalog for monitoring and debugging

&copy; 2026 Databricks, Inc. All rights reserved. Apache, Apache Spark, Spark, the Spark Logo, Apache Iceberg, Iceberg, and the Apache Iceberg logo are trademarks of the <a href="https://www.apache.org/" target="_blank">Apache Software Foundation</a>.<br/><br/><a href="https://databricks.com/privacy-policy" target="_blank">Privacy Policy</a> | <a href="https://databricks.com/terms-of-use" target="_blank">Terms of Use</a> | <a href="https://help.databricks.com/" target="_blank">Support</a>
